In [4]:
import requests
import pandas as pd
import numpy as np
import io
from bs4 import BeautifulSoup
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

In [ ]:
# ==========================================
# 1. OBSERVATORY LOCATION FUNCTION
# ==========================================
def get_observatory_location(iau_code):
    url_obs = "https://minorplanetcenter.net/iau/lists/ObsCodes.html"
    print(f"--- Downloading official MPC list to search for code {iau_code}... ---")
    r = requests.get(url_obs)
    r.raise_for_status()
    

    for line in r.text.split('\n'):
        if line.startswith(iau_code):
            parts = line.split()
            if len(parts) >= 4:
                long_deg = float(parts[1])
                cos_phi = float(parts[2])
                sin_phi = float(parts[3])
                lat_rad = np.arctan2(sin_phi, cos_phi)
                lat_deg = np.degrees(lat_rad)
                print(f"Observatory found: Lat {lat_deg:.4f}, Lon {long_deg:.4f}")
                return EarthLocation(lat=lat_deg * u.deg, lon=long_deg * u.deg, height=0 * u.m)
    
    print(f"Code {iau_code} not found.")
    print(long_deg , lat_def)
    return None

In [ ]:
# ==========================================
# 2. FILTERING FUNCTION
# ==========================================
def filter_visible_objects(df, location):
    #print("\n--- Calculating visibility (this may take a while) ---")
    visible_objects = []
    
    # Check next 24h (15 min steps)
    current_time = Time.now()
    delta_time = np.linspace(0, 24, 96) * u.hour 
    times_grid = current_time + delta_time
    
    frame_altaz = AltAz(obstime=times_grid, location=location)

    for index, row in df.iterrows():
        try:
            ra_raw = str(row['R.A.']).strip()
            dec_raw = str(row['Decl.']).strip()
            
            ra_txt = ra_raw.replace(" ", "h", 1) if "h" not in ra_raw else ra_raw
            if "m" not in ra_txt and "h" in ra_txt: ra_txt += "m"
            if "h" not in ra_txt: ra_txt += "h"

            dec_txt = dec_raw.replace(" ", "d", 1) if "d" not in dec_raw else dec_raw
            if "m" not in dec_txt and "d" in dec_txt: dec_txt += "m"
            if "d" not in dec_txt: dec_txt += "d"

            coord = SkyCoord(ra=ra_txt, dec=dec_txt, unit=(u.hourangle, u.deg))
            altaz = coord.transform_to(frame_altaz)
            altitudes = altaz.alt.degree
            
            # Logic: > 2 points (approx 30 mins) above 10 degrees
            visible_points = np.sum(altitudes > 10)
            
            if visible_points >= 2:
                row['Visible_Minutes'] = visible_points * 15
                row['Max_Alt'] = round(np.max(altitudes), 1)
                visible_objects.append(row)
                
        except Exception:
            continue 

    return pd.DataFrame(visible_objects)

In [ ]:
# ==========================================
# 3. PROCEESS MPC DATA
# ==========================================

def process_mpc_data(observatory_code, page_type="NEOCP", interactive_mode=True):
    """
    Downloads, processes, and filters MPC data (NEOCP or PCCP).
    
    Args:
        observatory_code (str): Observatory code (e.g., "Y28").
        page_type (str): "NEOCP" or "PCCP".
        interactive_mode (bool): If True, activates the loop to view details.
        
    Returns:
        pd.DataFrame: The filtered DataFrame with visible objects.
    """
    
    # 1. URL Configuration
    page_type = page_type.strip().upper()
    url = "https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html" if page_type == "NEOCP" else "https://minorplanetcenter.net/iau/NEO/pccp_tabular.html"

    print(f"--- Downloading data from: {page_type} ---")
    headers = {"User-Agent": "Mozilla/5.0"}
    df_raw = None

    # 2. Download and Parsing
    try:
        resp = requests.get(url, headers=headers)
        soup = BeautifulSoup(resp.content, 'html.parser')
        table_html = soup.find('table', {'class': 'tablesorter'})
        
        if table_html:
            # Clean hidden elements and checkboxes
            for hidden in table_html.find_all('span', style=lambda x: x and 'display:none' in x): hidden.decompose()
            for chk in table_html.find_all('input'): chk.decompose()
            
            # Read into DataFrame
            df_raw = pd.read_html(io.StringIO(str(table_html)), flavor='bs4')[0]
            df_raw.columns = [c.strip() for c in df_raw.columns]
            df_raw = df_raw.dropna(axis=1, how='all')
        else:
            print("HTML table not found.")

    except Exception as e:
        print(f"Download error: {e}")
        return pd.DataFrame() # Return empty DataFrame on error

    df_filtered = pd.DataFrame() 

    # 3. Filtering Logic
    if df_raw is not None and not df_raw.empty:
        # Note: Assumes get_observatory_location and filter_visible_objects 
        # are defined in your global scope or imported.
        local_obs = get_observatory_location(observatory_code)
        
        if local_obs is not None:
            print(f"\nTotal objects downloaded: {len(df_raw)}")
            print(f"Filtering objects with Alt > 10° for ~30 min at {observatory_code}...")
            
            df_filtered = filter_visible_objects(df_raw, local_obs)
            
            if not df_filtered.empty:
                df_filtered = df_filtered.sort_values(by='Max_Alt', ascending=False)
                
                # Display summary table
                cols = ['Temp Desig', 'R.A.', 'Decl.', 'V', 'Visible_Minutes', 'Max_Alt']
                final_cols = [c for c in cols if c in df_filtered.columns]
                
                print("\n" + "="*60)
                print(f"OBSERVABLE OBJECTS SUMMARY ({observatory_code})")
                print("="*60)
                # Temporary context to display all rows
                with pd.option_context('display.max_rows', None):
                    print(df_filtered[final_cols].to_string(index=False))
                
                # 4. Interactive Mode (Optional)
                if interactive_mode:
                    while True:
                        print("\n" + "-"*60)
                        target = input("Enter the 'Temp Desig' to see full details (or '0' to exit): ").strip()
                        
                        if target == '0':
                            print("Exiting interaction...")
                            break
                        
                        obj_row = df_filtered[df_filtered['Temp Desig'] == target]
                        
                        if not obj_row.empty:
                            print(f"\nDETAILS FOR OBJECT: {target}")
                            print("="*30)
                            print(obj_row.iloc[0]) 
                        else:
                            print(f"Object '{target}' not found via exact match. Check spelling.")

            else:
                print("\nNo objects visible with current criteria.")
        else:
            print("Failed to obtain observatory coordinates.")
    else:
        print("No data downloaded.")
        
    return df_filtered

In [ ]:
from cipo import process_mpc_data  # noqa: F401

In [ ]:
OBS_CODE = "Y28"
user_choice = input("Enter 'NEOCP' or 'PCCP' (Default: NEOCP): ").strip().upper()
page_type = "PCCP" if user_choice == "PCCP" else "NEOCP"

# Function Call
process_mpc_data(
    observatory_code=OBS_CODE, 
    page_type=page_type, 
    interactive_mode=True)

In [5]:
def mpc_obects(type_obj):
    """
    Busca a tabela de objetos do MPC (NEOCP ou PCCP), limpa os dados
    e imprime o DataFrame pandas completo.
    
    Parâmetros:
        type_obj (str): 'neocp' ou 'pccp'
    """
    # 1. Define a URL com base no parâmetro
    tipo = type_obj.lower().strip()
    
    if tipo == 'neocp':
        url = "https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html"
        
    elif tipo == 'pccp':
        url = "https://minorplanetcenter.net/iau/NEO/pccp_tabular.html"
        
    else:
        print("Erro: O parâmetro deve ser 'neocp' ou 'pccp'.")
        return

    # 2. Faz a requisição
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Verifica se deu erro 404/500
    except Exception as e:
        print(f"Erro de conexão: {e}")
        return

    # 3. Limpeza do HTML (BeautifulSoup)
    soup = BeautifulSoup(response.content, 'html.parser')
    tabela_html = soup.find('table', {'class': 'tablesorter'})

    if tabela_html:
        # Remove dados ocultos de ordenação (display:none)
        for hidden in tabela_html.find_all('span', style=lambda x: x and 'display:none' in x):
            hidden.decompose()
        
        # Remove checkboxes
        for checkbox in tabela_html.find_all('input'):
            checkbox.decompose()

        # 4. Converte para Pandas
        try:
            # Usa flavor='bs4' para evitar depender do lxml se ele não estiver instalado
            df_lista = pd.read_html(io.StringIO(str(tabela_html)), flavor='bs4')
            
            if df_lista:
                df = df_lista[0]
                
                # Ajustes finais da tabela
                df.columns = [c.strip() for c in df.columns] # Remove espaços dos nomes das colunas
                df = df.dropna(axis=1, how='all')            # Remove colunas vazias
                
                # Configuração de visualização para o print sair completo
                pd.set_option('display.max_rows', None)
                pd.set_option('display.max_columns', None)
                pd.set_option('display.width', 1000)
                pd.set_option('display.colheader_justify', 'left')

                print(f"\nTotal de objetos encontrados: {len(df)}")
                print("="*60)
                print(df)
                print("="*60)
            else:
                print("A tabela foi encontrada, mas o Pandas não conseguiu ler os dados.")
        except Exception as e:
            print(f"Erro na conversão para Pandas: {e}")
    else:
        print("Tabela não encontrada na página.")

In [6]:
mpc_objects('neocp')

--- Acessando NEOCP (https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html) ---

Total de objetos encontrados: 73
   Temp Desig  Score Discovery     R.A.     Decl.    V    Updated                Note  NObs  Arc   H     Not Seen/dys
0   A11yePp     98.0  2026 02 12.2  10 30.6  +06 17  17.4  Updated Feb. 12.49 UT  NaN  15.0  0.28  28.6   0.064      
1       NaN      NaN           NaN      NaN     NaN   NaN                    NaN  NaN   NaN   NaN   NaN     NaN      
2   ST26B30    100.0  2026 02 12.3  10 39.2  +01 06  19.4  Updated Feb. 12.39 UT    S   6.0  0.06  34.5   0.185      
3       NaN      NaN           NaN      NaN     NaN   NaN                    NaN  NaN   NaN   NaN   NaN     NaN      
4   6BB4C21    100.0  2026 02 12.2  11 28.0  -16 22  19.7  Updated Feb. 12.34 UT  NaN   8.0  0.08  24.7   0.218      
5       NaN      NaN           NaN      NaN     NaN   NaN                    NaN  NaN   NaN   NaN   NaN     NaN      
6   gb00821     98.0  2026 02 11.8  09 52.9  +25 32  

In [12]:
import requests
import pandas as pd
import io
from bs4 import BeautifulSoup

def mpc_osdbjects(type_obj):
    """
    1. Baixa a tabela (NEOCP ou PCCP).
    2. Imprime a tabela na tela.
    3. Retorna o DataFrame para interação posterior.
    """
    # --- 1. CONFIGURAÇÃO ---
    tipo = type_obj.lower().strip()
    if tipo == 'neocp':
        url = "https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html"
        print(f"--- Baixando dados da NEOCP ({url}) ---")
    elif tipo == 'pccp':
        url = "https://minorplanetcenter.net/iau/NEO/pccp_tabular.html"
        print(f"--- Baixando dados da PCCP ({url}) ---")
    else:
        print("Erro: Escolha entre 'neocp' ou 'pccp'.")
        return None

    # --- 2. REQUISIÇÃO ---
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"Erro de conexão: {e}")
        return None

    # --- 3. LIMPEZA (BEAUTIFULSOUP) ---
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', {'class': 'tablesorter'})

    if not table:
        print("Tabela não encontrada.")
        return None

    # Remove dados ocultos e checkboxes
    for hidden in table.find_all('span', style=lambda x: x and 'display:none' in x):
        hidden.decompose()
    for chk in table.find_all('input'):
        chk.decompose()

    # --- 4. CONVERSÃO E PRINT ---
    try:
        # Lê a tabela limpa
        df = pd.read_html(io.StringIO(str(table)), flavor='bs4')[0]
        
        # Ajusta colunas
        df.columns = [c.strip() for c in df.columns]
        df = df.dropna(axis=1, how='all')

        # === PASSO A: IMPRIMIR (Visualização) ===
        print(f"\nSucesso! {len(df)} objetos encontrados.")
        print("="*80)
        
        # Configura pandas para mostrar tudo no print
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        
        #print(df)
        #print("="*80)
#print("A tabela foi impressa acima e também retornada para a variável.\n")

        # === PASSO B: RETORNAR (Interação) ===
        return df

    except Exception as e:
        print(f"Erro ao processar dados: {e}")
        return None

In [11]:
mpc_objects('neocp')

--- Baixando dados da NEOCP (https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html) ---

Sucesso! 73 objetos encontrados.


,Temp Desig,Score,Discovery,R.A.,Decl.,V,Updated,Note,NObs,Arc,H,Not Seen/dys
0,A11yePp,98.0,2026 02 12.2,10 30.8,+06 13,17.4,Updated Feb. 12.49 UT,NaN,15.0,0.28,28.6,0.067
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ST26B30,100.0,2026 02 12.3,10 39.9,+01 00,19.4,Updated Feb. 12.39 UT,S,6.0,0.06,34.5,0.188
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6BB4C21,100.0,2026 02 12.2,11 28.0,-16 22,19.7,Updated Feb. 12.34 UT,NaN,8.0,0.08,24.7,0.221
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,gb00821,98.0,2026 02 11.8,09 52.9,+25 32,20.6,Updated Feb. 12.14 UT,NaN,10.0,0.34,29.9,0.406
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,A11yeOi,79.0,2026 02 11.0,10 52.0,-15 10,19.5,Updated Feb. 12.46 UT,NaN,29.0,1.25,23.7,0.244
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
def mpc_objects(obj_type):
    """
    1. Downloads the MPC table (NEOCP or PCCP).
    2. Prints the full table to the console.
    3. Returns the Pandas DataFrame for further interaction.
    
    Parameters:
        obj_type (str): 'neocp' or 'pccp'
    """
    #1. Configuration 
    target_type = obj_type.lower().strip()
    
    if target_type == 'neocp':
        url = "https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html"
        print(f"--- Downloading NEOCP data from: {url} ---")
    elif target_type == 'pccp':
        url = "https://minorplanetcenter.net/iau/NEO/pccp_tabular.html"
        print(f"--- Downloading PCCP data from: {url} ---")
    else:
        print("Error: Please specify 'neocp' or 'pccp'.")
        return None

     #2. Http Request 
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Check for 404/500 errors
    except Exception as e:
        print(f"Connection error: {e}")
        return None

    #3. Html Cleaning 
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', {'class': 'tablesorter'})

    if not table:
        print("Table not found on the page.")
        return None

    # Remove hidden sorting data 
    for hidden in table.find_all('span', style=lambda x: x and 'display:none' in x):
        hidden.decompose()
        
    # Remove checkboxes
    for checkbox in table.find_all('input'):
        checkbox.decompose()

    #4. Conversion and printing
    try:
        # Read the cleaned HTML table into Pandas
        # Using flavor='bs4' to avoid lxml dependency issues
        df_list = pd.read_html(io.StringIO(str(table)), flavor='bs4')
        
        if df_list:
            df = df_list[0]
            
            # Clean column names and drop empty columns
            df.columns = [c.strip() for c in df.columns]
            df = df.dropna(axis=1, how='all')

            # === STEP A: PRINT (Visualization) ===
            print(f"\nSuccess! {len(df)} objects found.")
            print("="*80)
            
            # Configure Pandas to display all rows and columns
            pd.set_option('display.max_rows', None)
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 1000)
            pd.set_option('display.colheader_justify', 'left')
            
            #print(df)
            #print("="*80)
            #print("The table has been printed above and returned to the variable.\n")

            #Step B: Return (Interaction)
            return df
            
        else:
            print("Pandas could not read the table data.")
            return None

    except Exception as e:
        print(f"Error processing data: {e}")
        return None

In [14]:
mpc_objects('neocp')

--- Downloading NEOCP data from: https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html ---

Success! 73 objects found.
   Temp Desig  Score Discovery     R.A.     Decl.    V    Updated                Note  NObs  Arc   H     Not Seen/dys
0   A11yePp     98.0  2026 02 12.2  10 30.8  +06 13  17.4  Updated Feb. 12.49 UT  NaN  15.0  0.28  28.6   0.067      
1       NaN      NaN           NaN      NaN     NaN   NaN                    NaN  NaN   NaN   NaN   NaN     NaN      
2   ST26B30    100.0  2026 02 12.3  10 39.9  +01 00  19.4  Updated Feb. 12.39 UT    S   6.0  0.06  34.5   0.188      
3       NaN      NaN           NaN      NaN     NaN   NaN                    NaN  NaN   NaN   NaN   NaN     NaN      
4   6BB4C21    100.0  2026 02 12.2  11 28.0  -16 22  19.7  Updated Feb. 12.34 UT  NaN   8.0  0.08  24.7   0.221      
5       NaN      NaN           NaN      NaN     NaN   NaN                    NaN  NaN   NaN   NaN   NaN     NaN      
6   gb00821     98.0  2026 02 11.8  09 52.9  +25

,Temp Desig,Score,Discovery,R.A.,Decl.,V,Updated,Note,NObs,Arc,H,Not Seen/dys
0,A11yePp,98.0,2026 02 12.2,10 30.8,+06 13,17.4,Updated Feb. 12.49 UT,NaN,15.0,0.28,28.6,0.067
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ST26B30,100.0,2026 02 12.3,10 39.9,+01 00,19.4,Updated Feb. 12.39 UT,S,6.0,0.06,34.5,0.188
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6BB4C21,100.0,2026 02 12.2,11 28.0,-16 22,19.7,Updated Feb. 12.34 UT,NaN,8.0,0.08,24.7,0.221
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,gb00821,98.0,2026 02 11.8,09 52.9,+25 32,20.6,Updated Feb. 12.14 UT,NaN,10.0,0.34,29.9,0.406
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,A11yeOi,79.0,2026 02 11.0,10 52.0,-15 10,19.5,Updated Feb. 12.46 UT,NaN,29.0,1.25,23.7,0.244
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
